# Recover a pipeline when one step drifts

In this guide, you'll run a short pipeline, catch a bad middle step, and retry from the last good
boundary. The retry cites the retained plan artifact and causally follows the failed draft and
inspector — it does not write into an existing output as if it were a branch. The sample task is
an infographic tile. The same pattern works for code generation, data transforms, migrations, and
other pipelines where one bad step can spoil useful work.

## Setup

Load the launch helpers.

In [ ]:
import pathlib
import sys


def _find_visual_artifact_example_root():
    cwd = pathlib.Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.append(base)
        candidates.append(base / "examples" / "notebooks" / "visual_artifact")
    for candidate in candidates:
        if (candidate / "shepherd_usecases" / "visual_artifact" / "launch.py").exists():
            return candidate
    raise RuntimeError(
        "Cannot find examples/notebooks/visual_artifact. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    )


example_root = _find_visual_artifact_example_root()
if str(example_root) not in sys.path:
    sys.path.insert(0, str(example_root))
try:
    from shepherd_usecases.visual_artifact import launch, viz
except Exception as exc:
    raise RuntimeError(
        "Could not import the visual-artifact notebook helpers. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    ) from exc

launch.bootstrap(example_root=example_root)

## What this run does

Pipeline Recovery runs a two-step pipeline: a plan, then a draft. A reviewer catches the draft
going wrong, an inspector reads the trace to diagnose it, and the draft is retried as a new run
that cites the retained plan artifact (`plan_ref`), the last good boundary.

## The input

For this example, the run starts from one plain prompt. `launch.plan_for` turns it into a `brief`
and an initial `plan`; both are used as inputs to the pipeline's first task.

In [ ]:
prompt = launch.default_prompt()
brief, plan = launch.plan_for(prompt)
plan

## Run Shepherd

First, open a workspace for the prompt. Every run in this guide belongs to it.

In [ ]:
workspace = launch.open_workspace("visual-pipeline-recovery", prompt=prompt, metadata={"usecase": "visual-pipeline-recovery"})

Now run the pipeline: first the plan, then the draft. The plan is the last good boundary; the draft
is the step that drifts.

In [ ]:
plan_run = launch.run_static(
    workspace,
    name="plan",
    output_path=launch.PLAN_PATH,
    output_content=plan,
    metadata={"logical_boundary": "plan"},
)
plan_ref = launch.artifact_ref(plan_run, launch.PLAN_PATH, label="retry-plan")

draft_v1 = launch.run_with_artifact_ref(
    workspace,
    name="draft-v1",
    ref_name="plan",
    artifact_ref=plan_ref,
    output_path=launch.ARTIFACT_PATH,
    output_text=launch.draft_html(brief, corrupt=True),
    after=[plan_run],
    metadata={"failed_run": "draft-v1"},
)
viz.show(viz.run_artifact(draft_v1, label="draft v1: wrong direction", accent="red"))

Two tasks examine the failed draft. First a reviewer flags it — an agent that reads the draft and
judges whether it met the brief. Then an inspector reads the reviewer's verdict and writes a
structured diagnosis: what drifted and what to change on retry. The reviewer's output is retained
read-only through `review_ref`; the inspector does not write into it.

In [ ]:
draft_ref = launch.artifact_ref(draft_v1, label="failed-draft")
reviewer = launch.run_with_artifact_ref(
    workspace,
    name="review",
    ref_name="candidate",
    artifact_ref=draft_ref,
    output_path=launch.VERDICT_PATH,
    output_content=launch.review_content(prompt, {"draft_v1": draft_v1}),
    after=[draft_v1],
)
selection = launch.selection_from_review(reviewer)
issues = list(selection.candidates[0].get("issues", []))
review_ref = launch.artifact_ref(reviewer, launch.VERDICT_PATH, label="draft-review")
inspector = launch.run_with_artifact_ref(
    workspace,
    name="inspector",
    ref_name="review",
    artifact_ref=review_ref,
    output_path=launch.DIAGNOSIS_PATH,
    output_content=launch.diagnosis_content(issues),
    after=[reviewer],
)
launch.read_json(inspector, launch.DIAGNOSIS_PATH)

The inspector's diagnosis is packaged as `diagnosis_ref`. The retry is a new run — not a branch
of an existing one. It cites both `plan_ref` (the retained plan artifact, the last good boundary)
and `diagnosis_ref` as artifact refs, and declares `after=[plan_run, inspector]` so the workspace
trace records the causal chain. The failed draft stays untouched as evidence.

In [ ]:
diagnosis_ref = launch.artifact_ref(inspector, launch.DIAGNOSIS_PATH, label="retry-diagnosis")
retry = launch.run_with_artifact_refs(
    workspace,
    name="retry",
    refs={"plan": plan_ref, "diagnosis": diagnosis_ref},
    output_path=launch.ARTIFACT_PATH,
    output_text=launch.draft_html(brief, corrupt=False),
    after=[plan_run, inspector],
    metadata={"retry_run": "retry-from-plan"},
)
viz.show(viz.side_by_side([
    viz.run_artifact(draft_v1, label="failed draft", accent="red"),
    viz.run_artifact(retry, label="retry from cited plan", accent="green"),
]))

The two draft outputs now sit side by side: the failed run is still available as evidence, and the
retry is the result of the new run with the fix applied. Now formalize the decision: discard the
failed draft, release the supporting runs, and select the retry.

In [ ]:
draft_v1.output().discard()
plan_run.output().release()
reviewer.output().release()
inspector.output().release()
retry.output().select()

## Trace

`workspace.flow` captures every action taken across all runs above, plus the causal relationships
among them — including the link from `plan_run` and `inspector` to `retry`. Click the nodes to
explore individual events.

In [ ]:
viz.show(viz.trace(
    workspace.flow,
    {"plan": plan_run, "draft_v1": draft_v1, "review": reviewer, "inspector": inspector, "retry": retry},
    height="680px",
    detail="events",
))

In [ ]:
workspace.close()

## Want to understand how it works?

If you want to understand how this works in greater detail, open the
[Pipeline Recovery internals notebook](visual_pipeline_recovery_internals.ipynb).

## Next steps

- [Find the best approach by trying several alternatives](visual_variant_studio.ipynb)
- [Find the cheapest model that still does the job](model_right_sizing_lab.ipynb)